# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adeenafatima0/ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

One row = one page, on one day, for one client. That's the grain of fact_content_daily_performance. I'm using March 2026 as my dev month since we're told not to build anything on the last month — that one's basically a sealed test set. Verifying the actual row count and dates below with a real query.

In [11]:
import duckdb, os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")
con.sql(f"""
CREATE SECRET hf_token (
    TYPE HUGGINGFACE,
    TOKEN '{os.environ["HF_TOKEN"]}'
);
""")

MARCH_PATH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"

## 2. Fields: feature / label / context / excluded

- **Features**: impressions, clicks, avg position, CTR, sessions, word count, content age, freshness — stuff that's actually observed and would've been known before I'd make any call on a page.
- **Label/proxy**: whether a page's performance dropped later on — I'll build this off the actual daily trend across the month, not some pre-made flag.
- **Context, not modeling input**: client_hash_id, content_hash_id — just ids to join tables, they don't mean anything as numbers themselves.
- **Excluded on purpose**: anything like health_score or priority_score. They're not even in this dataset, which is on purpose — if I used those, the model would basically just be copying FlyRank's existing decision instead of learning anything new from the raw signals.

In [12]:
import duckdb, os

con = duckdb.connect()
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")

# Properly register the HF token with DuckDB
con.sql(f"""
CREATE SECRET hf_token (
    TYPE HUGGINGFACE,
    TOKEN '{os.environ["HF_TOKEN"]}'
);
""")

df_march = con.sql("""
    SELECT *
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
    LIMIT 5
""").df()

df_march

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## 3. Verify it with queries (grain, counts, missing values, windows)

Three checks below: is the grain actually what I claimed (one row per page per day), how many rows/what date range am I actually working with, and how much of it has real usable data vs just "not tracked yet."

In [13]:
# check 1: is (date, client, content) actually unique per row?
grain_check = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS unique_combos
    FROM '{MARCH_PATH}'
""").df()
print(grain_check)

# check 2: how many rows, what dates does march actually cover?
span_check = con.sql(f"""
    SELECT COUNT(*) AS row_count, MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM '{MARCH_PATH}'
""").df()
print(span_check)

# check 3: how many rows actually have real GSC data, not just "tracked but empty"
availability_check = con.sql(f"""
    SELECT COUNT(*) AS rows_with_gsc_data
    FROM '{MARCH_PATH}'
    WHERE gsc_data_available IS TRUE
""").df()
print(availability_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  unique_combos
0     9841378        9841378
   row_count   min_date   max_date
0    9841378 2026-03-01 2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   rows_with_gsc_data
0             3611061


## 4. Data limits

Not every row in this data can actually be used. Out of the 9.84M rows in March 2026, only about 3.6M (roughly 37%) have real GSC data available — the rest are rows where a client's search tracking wasn't set up yet or wasn't syncing that day. If I don't filter on gsc_data_available, I'd be treating "we don't have this info yet" the same as "this page got zero visibility," which would badly skew any feature or label I build.

Also, since clients started tracking at different times (some clients have way more history than others), any time-based feature (like "impressions over prior 90 days") needs to check each client's actual tracking start date first — otherwise I might treat a gap in tracking as a real drop in performance, which it isn't.

Finally, since I'm only using March 2026 to build and test logic, any label depending on "what happens next" would need to look forward past March, into April — which means I need to be careful my feature window and target window never overlap.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.